# 🔬 Receita Básica — Processamento de Imagens Micro-FTIR com HSP 2026

**Objetivo:** processar uma imagem de Micro-FTIR do zero até obter um dado limpo e salvo, sem usar a correção de parafina/água (MSC).  
**Tempo estimado:** ~15 minutos  
**Nível:** iniciante — sem experiência prévia com HSP

---

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tmpereira/hsp_2026/blob/main/docs/receitas/receita_basica.ipynb)  &nbsp;&nbsp; [![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-blue?logo=jupyter)](https://raw.githubusercontent.com/tmpereira/hsp_2026/main/docs/receitas/receita_basica.ipynb)

---


## O que vamos fazer nesta receita?

Imagine que você acabou de sair do microscópio com um arquivo de imagem Micro-FTIR. O que fazer com ele? Esta receita guia você passo a passo:

1. **Instalar e importar** a biblioteca HSP 2026  
2. **Ler** o arquivo bruto do equipamento  
3. **Entender** a estrutura interna dos dados  
4. **Visualizar** a imagem para saber o que você está vendo  
5. **Cortar** a região espectral de interesse (jogar fora o que não serve)  
6. **Remover pixels ruins** com Quality Test  
7. **Salvar** o dado processado para uso futuro  

---

> **Nota sobre equipamentos:** esta receita mostra exemplos para os dois equipamentos mais comuns:
> - **Perkin-Elmer Spotlight** → arquivos `.fsm`
> - **Agilent Cary 620/670** → arquivos `.dmd` + `.dmt`
>
> Depois que você leu a imagem, o restante do processamento é **idêntico** para os dois equipamentos.

---
## Passo 0 — Instalar a biblioteca HSP 2026

A biblioteca não está no PyPI, mas fica no GitHub. A célula abaixo faz a instalação diretamente de lá.  
**Execute esta célula uma única vez** (se você estiver rodando no Google Colab, ela já será suficiente — não é preciso instalar nada no seu computador).

In [ ]:
# Instala a biblioteca HSP 2026 direto do GitHub
# (precisa rodar apenas uma vez por sessão no Colab)
!pip install git+https://github.com/tmpereira/hsp_2026.git -q

---
## Passo 1 — Importar os módulos

A biblioteca HSP 2026 é organizada em **submódulos**, cada um com uma responsabilidade clara:

| Submódulo | O que faz |
|-----------|----------|
| `file`    | Lê e salva arquivos de imagem |
| `sh`      | Visualiza imagens (mapas de cor) |
| `prep`    | Pré-processa espectros (corte, normalização, filtros) |
| `rg`      | Gera histogramas para diagnóstico |
| `qt`      | Faz Quality Test (remove pixels ruins) |
| `km`      | Clusterização K-means (imagens de falsa cor) |
| `emsc`    | Correção de parafina e vapor d'água (MSC) |

Nesta receita básica vamos usar apenas `file`, `sh`, `prep`, `rg` e `qt`.

In [ ]:
import hsp_2026.file as file
import hsp_2026.sh   as sh
import hsp_2026.prep as prep
import hsp_2026.rg   as rg
import hsp_2026.qt   as qt

print('✅ Módulos importados com sucesso!')

---
## Passo 2 — Ler o arquivo de imagem

### O que é esse arquivo?

Quando o microscópio faz uma medição, ele gera um arquivo binário com o "cubo de dados": cada pixel da imagem carrega um espectro completo de infravermelho. Pense assim:

```
Imagem 170 × 204 pixels
  └── cada pixel = 1 espectro com ~1600 pontos de absorbância
      └── eixo X: número de onda (cm⁻¹)
          eixo Y: absorbância
```

### Como ler?

Escolha a função correta conforme o equipamento que gerou seu arquivo:

In [ ]:
# ── OPÇÃO A: Perkin-Elmer Spotlight (arquivo .fsm) ──────────────────────────
#
# A entrada é o caminho completo para o arquivo .fsm.
# No Google Colab, faça upload do arquivo clicando no ícone de pasta (📁)
# na barra lateral esquerda e copie o caminho.

caminho_arquivo = 'minha_amostra.fsm'   # ← altere aqui
data = file.fsm(caminho_arquivo)

# ── OPÇÃO B: Agilent Cary 620/670 ────────────────────────────────────────────
#
# ATENÇÃO: para o Agilent, a entrada NÃO é um arquivo — é a PASTA que contém
# todos os arquivos da imagem (.dmd e .dmt). O equipamento divide a imagem em
# vários tiles e salva cada um como um .dmd separado; a função lê todos de uma vez.
#
# Estrutura esperada da pasta:
#   minha_pasta/
#     ├── 0000_0000.dmd
#     ├── 0000_0001.dmd
#     ├── 0001_0000.dmd
#     ├── ...           (quantos tiles tiver a imagem)
#     └── arquivo.dmt   (metadados espectrais)
#
# A barra final '/' no caminho da pasta é obrigatória.
#
# caminho_pasta = '/caminho/para/minha_pasta/'   # ← altere aqui
# data = file.age(caminho_pasta)

print('✅ Arquivo lido com sucesso!')
print(f'   Imagem: {data["dx"]} × {data["dy"]} pixels')
print(f'   Espectros: {data["r"].shape[0]} pixels × {data["r"].shape[1]} pontos espectrais')


---
## Passo 3 — Entender a estrutura dos dados

Após a leitura, `data` é um **dicionário Python** com sempre as mesmas chaves, independente do equipamento usado. Essa padronização é proposital: qualquer função de processamento sabe exatamente onde encontrar o que precisa.

| Chave | Tipo | O que armazena |
|-------|------|----------------|
| `r`   | matriz float (n_pixels × n_pontos) | Absorbâncias de todos os pixels |
| `wn`  | vetor float (n_pontos,) | Números de onda em cm⁻¹ |
| `dx`  | int | Número de **linhas** da imagem |
| `dy`  | int | Número de **colunas** da imagem |
| `sel` | vetor booleano (n_pixels,) | Máscara: `True` = pixel válido |
| `filename` | string | Nome do arquivo de origem |
| `log` | string | Histórico das operações feitas |

> **Dica:** `dx × dy` sempre é igual ao número de linhas em `r`. Por exemplo, uma imagem de 170 × 204 pixels tem 34.680 espectros.

In [ ]:
# Vamos explorar o dicionário
print('Chaves disponíveis:', list(data.keys()))
print()
print(f'Dimensões da imagem  → dx={data["dx"]} linhas, dy={data["dy"]} colunas')
print(f'Total de pixels      → {data["dx"] * data["dy"]} (= {data["dx"]} × {data["dy"]})')
print(f'Tamanho de data["r"] → {data["r"].shape} (pixels × pontos espectrais)')
print()
print(f'Faixa espectral: {data["wn"].min():.0f} até {data["wn"].max():.0f} cm⁻¹')
print()
print('Histórico (log):')
print(data['log'])

---
## Passo 4 — Visualizar a imagem bruta

Antes de qualquer processamento, é importante **ver** a imagem para avaliar a qualidade
do dado coletado em cada pixel.

### Por que usar a área integrada em vez da intensidade em um único número de onda?

A função `sh.intt()` mostra a intensidade em um único ponto do espectro.
Na prática, ela é sensível a ruídos pontuais e pode gerar imagens com contraste ruim.
A função `sh.area()` calcula a **área sob a curva** numa faixa inteira, o que:

- **Reduz o efeito do ruído** — a área integra muitos pontos, não depende de um único
- **Gera imagens mais nítidas e com melhor contraste**
- **Deixa imediatamente visível** onde há sinal biológico e onde não há

### Quais faixas usar?

| Faixa (cm⁻¹) | O que representa | Quando usar |
|--------------|-----------------|-------------|
| **900 – 1800** | Sinal total da região de impressão digital | Avaliação geral da qualidade do espectro |
| **1500 – 1700** | Amida I + Amida II (proteínas) | Distinguir tecido de parafina/substrato |

### Como interpretar o mapa de área das Amidas (1500–1700 cm⁻¹)?

A parafina tem absorção muito fraca nessa região, enquanto o tecido absorve fortemente
(bandas de Amida I ~1650 cm⁻¹ e Amida II ~1550 cm⁻¹). O resultado é um mapa de duas cores bem definidas:

- **Pixels claros/quentes** → presença de tecido
- **Pixels escuros/frios** → parafina, substrato vazio ou borda da amostra

Esse contraste é o mesmo que usaremos no **Quality Test (Passo 7)** para decidir
quais pixels manter.

> **Nota:** `sh.int_plt()` gera uma imagem interativa onde você pode clicar num pixel
> para ver seu espectro. Funciona no Spyder (com `%matplotlib qt`), mas
> **não funciona no Google Colab**. As funções `sh.area()` e `sh.intt()` funcionam nos dois ambientes.


In [ ]:
# Mapa de área total (900–1800 cm⁻¹)
# Mostra a qualidade geral do sinal em cada pixel:
# pixels com boa coleta aparecem brilhantes, regiões sem sinal aparecem escuras.
sh.area(data, 900, 1800)


In [ ]:
# Mapa de área das Amidas (1500–1700 cm⁻¹)
# Esta é a visualização mais útil para amostras embebidas em parafina:
# o contraste entre tecido (claro) e parafina/fundo (escuro) fica muito evidente,
# porque a parafina quase não absorve nessa faixa.
#
# Use este mapa para:
#   1. Avaliar se a amostra foi bem coletada
#   2. Identificar visualmente as regiões de tecido
#   3. Guiar a escolha do limiar no Quality Test (Passo 7)
sh.area(data, 1500, 1700)


---
## Passo 5 — Corte espectral (restrição de faixa)

O equipamento coleta dados de ~450 até ~4000 cm⁻¹, mas a região realmente útil para análise de tecidos é muito menor. Fora dessa faixa temos:

- **Abaixo de 900 cm⁻¹:** ruído do detector, poucos sinais biológicos relevantes  
- **Acima de 1800 cm⁻¹:** bandas de água e CO₂ do ar que poluem o sinal  

O corte tem dois benefícios:
1. **Reduz o tamanho do arquivo** — de ~1600 para ~550 pontos  
2. **Foca o processamento** onde o sinal biológico está  

**Faixas típicas:**
- `900–1800 cm⁻¹` → uso geral (proteínas, carboidratos, lipídios)  
- `900–1350 cm⁻¹` → foco em carboidratos e ácidos nucleicos

In [ ]:
# Corte espectral: mantém apenas a faixa de 900 a 1800 cm⁻¹
# (usamos 800 e 1900 para garantir que os extremos sejam incluídos)
data = prep.cut(data, 800, 1900)

# Confirma a nova faixa espectral
print(f'Nova faixa espectral: {data["wn"].min():.0f} – {data["wn"].max():.0f} cm⁻¹')
print(f'Novos pontos espectrais: {data["r"].shape[1]} (antes havia mais de 1600)')

# O log registra automaticamente o que foi feito
print('\nLog atualizado:')
print(data['log'])

---
## Passo 6 — Diagnóstico com histograma

Antes de remover pixels, precisamos **decidir o limiar de corte** de forma inteligente. O histograma de área é a ferramenta para isso.

### O que é a "área integrada"?

Para cada pixel, calculamos a área sob a curva do espectro na faixa da **Amida I** (1600–1800 cm⁻¹). Essa área é proporcional à quantidade de proteína naquele pixel:

- **Pixels com pouca área** → não há tecido ali (apenas parafina ou substrato vazio)  
- **Pixels com muita área** → presença de tecido (proteínas absorvem fortemente em ~1650 cm⁻¹)  

O histograma mostra a distribuição de todos os pixels: você verá dois grupos bem separados.

In [ ]:
# Histograma da área na banda de Amida I
# Use este gráfico para escolher o valor mínimo de área no Passo 7
rg.area(data, 1600, 1800)

# Como interpretar:
# • Um pico grande PRÓXIMO de zero → pixels de parafina/fundo (a remover)
# • Um segundo pico com valores maiores → pixels de tecido (a manter)
# • O limiar de corte deve cair no "vale" entre os dois picos

---
## Passo 7 — Quality Test: remover pixels ruins

Com o histograma em mãos, agora é hora de remover os pixels que não têm tecido.  
Este passo é chamado de **Quality Test (QT)**.

### Como definir os limites?

Olhe o histograma gerado no Passo 6:
- **Valor mínimo (`a`):** valor no "vale" entre o pico de parafina e o pico de tecido. Um valor comum para começar é `8`, mas ajuste conforme seu histograma.
- **Valor máximo (`b`):** pode deixar bem alto (ex: `10000`) para não remover pixels intensos demais. Só ajuste se houver evidência de saturação.

> **Regra geral:** comece com `a = 8` e `b = 10000`. Se ainda restar muita parafina na imagem, aumente `a`. Se perder tecido, diminua `a`.

In [ ]:
# Quality Test baseado na área da banda de Amida I
#
# Parâmetros:
#   data        → dicionário hsp
#   1600, 1800  → faixa espectral para calcular a área (Amida I)
#   8           → valor mínimo de área (pixels abaixo disso são removidos)
#   10000       → valor máximo de área (praticamente sem limite superior)

data = qt.area(data, 1600, 1800, 8, 10000)

print(f'\nEspectros restantes: {data["r"].shape[0]}')
print(f'Pixels válidos (sel=True): {data["sel"].sum()}')

In [ ]:
# Visualiza a imagem após o Quality Test
# Os pixels removidos aparecem como fundo escuro (zero)
sh.intt(data, 1650)

# Compare com a imagem bruta do Passo 4:
# as bordas de parafina e regiões sem tecido devem ter sumido

In [ ]:
# Opcionalmente: visualize também pela área (mostra o "mapa de proteína")
sh.area(data, 1600, 1800)

---
## Passo 8 — Salvar o dado processado

Agora que o dado está limpo e pré-processado, vamos salvá-lo. O formato usado é o `.npz`, que é o formato nativo do NumPy — compacto e rápido de carregar.

**Por que salvar?**  
As etapas de leitura e Quality Test podem demorar para imagens grandes. Salvando o resultado, nas próximas vezes que precisar dos dados você pode pular direto para a análise.

In [ ]:
# Salva o dado processado
# O arquivo será criado na pasta atual com o nome escolhido
nome_arquivo_saida = 'minha_amostra_processada.npz'   # ← altere à vontade

file.npz_save(nome_arquivo_saida, data)

print(f'✅ Arquivo salvo como: {nome_arquivo_saida}')
print('   Use file.npz_load() para carregar novamente nas próximas sessões.')

---
## (Bônus) Como recarregar um arquivo já processado

Na próxima vez que quiser trabalhar com esses dados, você não precisa repetir todos os passos. Basta carregar o arquivo `.npz`:

In [ ]:
# Recarrega o dado salvo
data_recarregado = file.npz_load('minha_amostra_processada.npz')

print('✅ Arquivo recarregado!')
print(f'   Espectros: {data_recarregado["r"].shape[0]}')
print(f'   Pontos espectrais: {data_recarregado["r"].shape[1]}')
print('\nHistórico de processamento:')
print(data_recarregado['log'])

---
## Resumo do fluxo completo

```python
import hsp_2026.file as file
import hsp_2026.sh   as sh
import hsp_2026.prep as prep
import hsp_2026.rg   as rg
import hsp_2026.qt   as qt

# 1. Ler o arquivo
data = file.fsm('minha_amostra.fsm')        # Perkin-Elmer
# data = file.age('minha_amostra.dmd')      # Agilent

# 2. Visualizar a imagem bruta
sh.intt(data, 1650)

# 3. Corte espectral
data = prep.cut(data, 800, 1900)

# 4. Diagnóstico com histograma → escolher limiar
rg.area(data, 1600, 1800)

# 5. Quality Test
data = qt.area(data, 1600, 1800, 8, 10000)

# 6. Visualizar resultado
sh.intt(data, 1650)

# 7. Salvar
file.npz_save('minha_amostra_processada.npz', data)
```

---

## Próximos passos

Agora que você tem um dado limpo, você pode:

- **Fazer imagens de falsa cor** → `km.py` (clusterização K-means)  
- **Normalizar os espectros** → `prep.norm()` ou `prep.snv()`  
- **Suavizar ruídos** → `prep.golay()` (filtro Savitzky-Golay)  
- **Corrigir parafina e vapor d'água** → receita com `emsc.py` (MSC)  

Consulte a [documentação completa](https://tmpereira.github.io/hsp_2026) para detalhes de cada módulo.